In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated successfully!")

✅ Authenticated successfully!


In [ ]:
!pip install gcsfs pyarrow pandas -q
print("✅ Libraries installed!")

✅ Libraries installed!


In [ ]:
import urllib.request
import os

os.makedirs("/content/raw_data", exist_ok=True)

files = {
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2024-02.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet",
}

for filename, url in files.items():
    print(f"⬇️ Downloading {filename}...")
    urllib.request.urlretrieve(url, f"/content/raw_data/{filename}")
    print(f"✅ Done: {filename}")

print("\n🎉 All files downloaded!")

⬇️ Downloading yellow_tripdata_2024-01.parquet...
✅ Done: yellow_tripdata_2024-01.parquet
⬇️ Downloading yellow_tripdata_2024-02.parquet...
✅ Done: yellow_tripdata_2024-02.parquet

🎉 All files downloaded!


In [ ]:
for f in os.listdir("/content/raw_data"):
    size_mb = os.path.getsize(f"/content/raw_data/{f}") / (1024 * 1024)
    print(f"{f} — {size_mb:.1f} MB")

yellow_tripdata_2024-02.parquet — 48.0 MB
yellow_tripdata_2024-01.parquet — 47.6 MB


In [ ]:
import pandas as pd

df = pd.read_parquet("/content/raw_data/yellow_tripdata_2024-01.parquet")

print("✅ Shape:", df.shape)
print("\n📋 Columns:", df.columns.tolist())
print("\n🔢 Data Types:\n", df.dtypes)
print("\n🔍 Sample rows:")
df.head(3)

✅ Shape: (2964624, 19)

📋 Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']

🔢 Data Types:
 VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge 

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0


In [ ]:
print("❌ Null counts:\n", df.isnull().sum())
print("\n📊 Basic stats:")
df[["trip_distance", "total_amount", "passenger_count"]].describe()

❌ Null counts:
 VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          140162
trip_distance                 0
RatecodeID               140162
store_and_fwd_flag       140162
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     140162
Airport_fee              140162
dtype: int64

📊 Basic stats:


,trip_distance,total_amount,passenger_count
count,2.964624e+06,2.964624e+06,2.824462e+06
mean,3.652169e+00,2.680150e+01,1.339281e+00
std,2.254626e+02,2.338558e+01,8.502817e-01
min,0.000000e+00,-9.000000e+02,0.000000e+00
25%,1.000000e+00,1.538000e+01,1.000000e+00
50%,1.680000e+00,2.010000e+01,1.000000e+00
75%,3.110000e+00,2.856000e+01,1.000000e+00
max,3.127223e+05,5.000000e+03,9.000000e+00


In [ ]:
from google.cloud import storage
import os

# ⚠️ Replace with your GCP project ID and bucket name
PROJECT_ID = "nyc-tlc-pipeline"
BRONZE_BUCKET = "nyc-tlc-bronze-prasad"  # e.g. nyc-tlc-bronze-prasad

# Connect to GCS
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BRONZE_BUCKET)

# Upload all parquet files
raw_dir = "/content/raw_data"

for filename in os.listdir(raw_dir):
    if filename.endswith(".parquet"):
        blob_path = f"yellow_taxi/2024/{filename}"
        file_path = os.path.join(raw_dir, filename)

        print(f"⬆️ Uploading {filename} → bronze/{blob_path} ...")
        blob = bucket.blob(blob_path)
        blob.upload_from_filename(file_path)
        print(f"✅ Uploaded: {filename}")

print("\n🎉 All files uploaded to GCS Bronze bucket!")

⬆️ Uploading yellow_tripdata_2024-02.parquet → bronze/yellow_taxi/2024/yellow_tripdata_2024-02.parquet ...
✅ Uploaded: yellow_tripdata_2024-02.parquet
⬆️ Uploading yellow_tripdata_2024-01.parquet → bronze/yellow_taxi/2024/yellow_tripdata_2024-01.parquet ...
✅ Uploaded: yellow_tripdata_2024-01.parquet

🎉 All files uploaded to GCS Bronze bucket!


In [ ]:
!pip uninstall pyspark -y -q
!pip install pyspark==4.0.0 gcsfs google-cloud-storage -q
print("✅ Libraries installed!")

✅ Libraries installed!


In [ ]:
from google.colab import auth
auth.authenticate_user()

import os, subprocess
# Write credentials to standard location so gcsfs can find them
result = subprocess.run(
    ["gcloud", "auth", "application-default", "print-access-token"],
    capture_output=True, text=True
)
print("✅ Authenticated!" if result.returncode == 0 else "❌ Auth failed")

✅ Authenticated!


In [ ]:
# ⚠️ Replace with YOUR actual bucket names
PROJECT_ID    = "nyc-tlc-pipeline"
BRONZE_BUCKET = "nyc-tlc-bronze-prasad"   # e.g. nyc-tlc-bronze-prasad
SILVER_BUCKET = "nyc-tlc-silver-prasad"   # e.g. nyc-tlc-silver-prasad

BRONZE_PATH = f"gs://{BRONZE_BUCKET}/yellow_taxi/2024/"
SILVER_PATH = f"gs://{SILVER_BUCKET}/yellow_taxi/2024/"

print(f"✅ Bronze: {BRONZE_PATH}")
print(f"✅ Silver: {SILVER_PATH}")

✅ Bronze: gs://nyc-tlc-bronze-prasad/yellow_taxi/2024/
✅ Silver: gs://nyc-tlc-silver-prasad/yellow_taxi/2024/


In [ ]:
from google.cloud import storage

client = storage.Client(project=PROJECT_ID)
blobs = list(client.bucket(BRONZE_BUCKET).list_blobs())

print(f"✅ Files in Bronze bucket: {len(blobs)}")
for b in blobs:
    print(f"  📄 {b.name} — {b.size/(1024*1024):.1f} MB")

✅ Files in Bronze bucket: 2
  📄 yellow_taxi/2024/yellow_tripdata_2024-01.parquet — 47.6 MB
  📄 yellow_taxi/2024/yellow_tripdata_2024-02.parquet — 48.0 MB


In [ ]:
# Instead of Spark reading GCS directly (which needs the JAR),
# we download to Colab's local /content/ folder first
import os
from google.cloud import storage

os.makedirs("/content/bronze", exist_ok=True)
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BRONZE_BUCKET)

blobs = list(bucket.list_blobs(prefix="yellow_taxi/2024/"))
for blob in blobs:
    filename = blob.name.split("/")[-1]
    local_path = f"/content/bronze/{filename}"
    print(f"⬇️ Downloading {filename}...")
    blob.download_to_filename(local_path)
    size_mb = os.path.getsize(local_path) / (1024*1024)
    print(f"✅ {filename} — {size_mb:.1f} MB")

print("\n🎉 All files downloaded to /content/bronze/")

⬇️ Downloading yellow_tripdata_2024-01.parquet...
✅ yellow_tripdata_2024-01.parquet — 47.6 MB
⬇️ Downloading yellow_tripdata_2024-02.parquet...
✅ yellow_tripdata_2024-02.parquet — 48.0 MB

🎉 All files downloaded to /content/bronze/


In [ ]:
from pyspark.sql import SparkSession

# Stop any existing session
try:
    spark.stop()
    print("Stopped old session")
except:
    pass

# Simple local Spark — no GCS connector needed
spark = SparkSession.builder \
    .appName("NYC-TLC-Silver") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark started! Version: {spark.version}")

Stopped old session
✅ Spark started! Version: 4.0.0


In [ ]:
# Read from local path — no GCS connector needed
df_bronze = spark.read.parquet("/content/bronze/")

print(f"✅ Bronze data loaded!")
print(f"📊 Total rows   : {df_bronze.count():,}")
print(f"📋 Total columns: {len(df_bronze.columns)}")
df_bronze.printSchema()

✅ Bronze data loaded!
📊 Total rows   : 5,972,150
📋 Total columns: 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [ ]:
from pyspark.sql.functions import col, count, when

print("=== NULL COUNTS ===")
df_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
]).show()

print("=== BASIC STATS ===")
df_bronze.select(
    "trip_distance", "total_amount",
    "passenger_count", "tip_amount"
).describe().show()

=== NULL COUNTS ===
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         325772|            0|    325772|            325772|           0|           0|           0|          0|   

In [ ]:
from pyspark.sql.functions import (
    col, round, hour, dayofweek,
    month, year, unix_timestamp, when, coalesce, lit
)

df_silver = df_bronze \
    .dropDuplicates() \
    .filter(
        (col("trip_distance") > 0) &
        (col("trip_distance") < 500) &
        (col("total_amount") > 0) &
        (col("total_amount") < 5000) &
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull()) &
        (col("PULocationID").isNotNull()) &
        (col("DOLocationID").isNotNull())
    ) \
    .withColumn("trip_duration_min",
        round((unix_timestamp("tpep_dropoff_datetime") -
               unix_timestamp("tpep_pickup_datetime")) / 60, 2)) \
    .filter(
        (col("trip_duration_min") > 0) &
        (col("trip_duration_min") < 300)
    ) \
    .withColumn("revenue_per_mile",
        round(col("total_amount") / col("trip_distance"), 2)) \
    .withColumn("pickup_hour",  hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day",   dayofweek("tpep_pickup_datetime")) \
    .withColumn("pickup_month", month("tpep_pickup_datetime")) \
    .withColumn("pickup_year",  year("tpep_pickup_datetime")) \
    .withColumn("is_weekend",
        when(dayofweek("tpep_pickup_datetime").isin([1, 7]), True)
        .otherwise(False)) \
    .withColumn("tip_percentage",
        round((col("tip_amount") / col("total_amount")) * 100, 2)) \
    .select(
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        coalesce(col("passenger_count"), lit(1)).cast("integer").alias("passenger_count"),  # ← fix
        "trip_distance",
        "PULocationID",
        "DOLocationID",
        col("payment_type").cast("integer"),
        "fare_amount",
        "tip_amount",
        "total_amount",
        "trip_duration_min",
        "revenue_per_mile",
        "pickup_hour",
        "pickup_day",
        "pickup_month",
        "pickup_year",
        "is_weekend",
        "tip_percentage"
    )

bronze_count = df_bronze.count()
silver_count = df_silver.count()
print(f"✅ Transformation complete!")
print(f"📊 Bronze rows : {bronze_count:,}")
print(f"📊 Silver rows : {silver_count:,}")
print(f"🗑️  Rows removed: {bronze_count - silver_count:,}")

✅ Transformation complete!
📊 Bronze rows : 5,972,150
📊 Silver rows : 5,774,598
🗑️  Rows removed: 197,552


In [ ]:
print("=== NULL CHECK (all should be 0) ===")
df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).show()

print("=== ENGINEERED COLUMN STATS ===")
df_silver.select(
    "trip_duration_min", "revenue_per_mile", "tip_percentage"
).describe().show()

print("=== SAMPLE ROWS ===")
df_silver.show(5, truncate=False)

=== NULL CHECK (all should be 0) ===
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+-----------------+----------------+-----------+----------+------------+-----------+----------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|trip_duration_min|revenue_per_mile|pickup_hour|pickup_day|pickup_month|pickup_year|is_weekend|tip_percentage|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+------------+-----------+----------+------------+-----------------+----------------+-----------+----------+------------+-----------+----------+--------------+
|       0|                   0|                    0|              0|            0|           0|           0|           0|          0|         0|           0| 

In [ ]:
import os

# Step 1: Write Silver parquet locally first
os.makedirs("/content/silver", exist_ok=True)

df_silver.write \
    .mode("overwrite") \
    .partitionBy("pickup_year", "pickup_month") \
    .parquet("/content/silver/")

print("✅ Silver written locally!")

# Step 2: Upload all silver files to GCS
from google.cloud import storage

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(SILVER_BUCKET)

uploaded = 0
for root, dirs, files in os.walk("/content/silver/"):
    for file in files:
        if file.endswith(".parquet"):
            local_path = os.path.join(root, file)
            # Preserve partition folder structure
            gcs_path = local_path.replace("/content/silver/", "yellow_taxi/2024/")
            blob = bucket.blob(gcs_path)
            blob.upload_from_filename(local_path)
            uploaded += 1
            print(f"  ☁️ Uploaded: {gcs_path}")

print(f"\n✅ {uploaded} files uploaded to GCS Silver bucket!")

✅ Silver written locally!
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2002/pickup_month=12/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00000-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00002-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00004-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=2024/pickup_month=2/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet
  ☁️ Uploaded: yellow_taxi/2024/pickup_year=

In [ ]:
from google.cloud import storage

client = storage.Client(project=PROJECT_ID)
blobs = list(client.bucket(SILVER_BUCKET).list_blobs())

print(f"✅ Files in Silver bucket: {len(blobs)}")
for b in blobs:
    print(f"  📄 {b.name} — {b.size/1024:.1f} KB")

✅ Files in Silver bucket: 24
  📄 yellow_taxi/2024/pickup_year=2002/pickup_month=12/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 4.9 KB
  📄 yellow_taxi/2024/pickup_year=2008/pickup_month=12/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 4.9 KB
  📄 yellow_taxi/2024/pickup_year=2009/pickup_month=1/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 4.9 KB
  📄 yellow_taxi/2024/pickup_year=2009/pickup_month=1/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 4.9 KB
  📄 yellow_taxi/2024/pickup_year=2009/pickup_month=1/part-00004-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 5.1 KB
  📄 yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00000-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 4.9 KB
  📄 yellow_taxi/2024/pickup_year=2023/pickup_month=12/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 5.1 KB
  📄 yellow_taxi/2024/pickup_year=2023/pickup_month=12/par